# Part 2 - Setting Up Python Virtual Environments on the SCC

The SCC's `module load python3` gives you a shared, system-wide Python installation with common packages already available. But for most real projects you'll want your **own** set of packages (and versions) that won't conflict with other users or other projects. That's what a **virtual environment** is for.

## Why Use a Virtual Environment?
- Different projects may need different (even conflicting) versions of the same package.
- You may need to `pip install` a package that isn't part of the shared module - the shared Python install is read-only to regular users.
- It makes your work **reproducible**: you can recreate the exact same set of packages on another machine (or share it with a labmate) using a requirements file.

## Option 1: `venv` (Python's Built-in Tool)
After loading a Python module, use `venv` to create an isolated environment in a folder of your choice:

```bash
module load python3/3.12.4

# Create the environment (only needs to be done once)
python3 -m venv ~/envs/energize_env

# Activate it (needs to be done every session, after loading the module)
source ~/envs/energize_env/bin/activate

# Now pip installs go into your private environment, not the shared install
pip install numpy pandas matplotlib scikit-learn

# When you're done
deactivate
```

Once activated, your shell prompt usually changes to show the environment name, e.g. `(energize_env) $`. Any `python` or `pip` command now uses this environment.

## Option 2: Conda / Miniconda
Conda is popular in the scientific Python community because it can also manage non-Python dependencies (like CUDA libraries). On the SCC:

```bash
module load miniconda/23.11

# Create an environment with a specific Python version and packages
conda create -n energize_env python=3.12 numpy pandas matplotlib scikit-learn -y

# Activate it
conda activate energize_env

# Install additional packages later
conda install pytorch -c pytorch
# or
pip install some-package

# When you're done
conda deactivate
```

Check `module avail miniconda` (or `anaconda`) on the SCC for the exact module name and version to load.

## Reproducibility with a Requirements File
To let others recreate your exact environment, export the list of installed packages:

```bash
# venv / pip
pip freeze > requirements.txt

# Recreate elsewhere
pip install -r requirements.txt
```

```bash
# conda
conda env export > environment.yml

# Recreate elsewhere
conda env create -f environment.yml
```

Commit `requirements.txt` or `environment.yml` alongside your code (e.g., in a Git repository) so your analysis can be reproduced months later, by you or by a collaborator.

## Using Your Environment in a Batch Job
Activate your environment inside the `.qsub` script, before running your script, just as you would interactively:

```bash
#!/bin/bash -l
#$ -N my_job
#$ -l h_rt=00:30:00
#$ -j y

module load python3/3.12.4
source ~/envs/energize_env/bin/activate

python my_analysis.py

deactivate
```

## Using Your Environment as a Jupyter Kernel
To use your virtual environment inside a Jupyter notebook (including on SCC OnDemand, see the next notebook), register it as a Jupyter **kernel**:

```bash
source ~/envs/energize_env/bin/activate
pip install ipykernel
python -m ipykernel install --user --name energize_env --display-name "Python (energize_env)"
```

After this, "Python (energize_env)" will appear as a selectable kernel in Jupyter.

### *Exercise*
1. On the SCC, create a `venv` environment called `energize_env` and install `numpy`, `pandas`, and `matplotlib` into it.
2. Export it to a `requirements.txt` file.
3. Register it as a Jupyter kernel, and confirm it appears when you open a notebook through SCC OnDemand.